In [1]:
# Colab (celda):     !pip -q install google-genai
# VS Code (terminal): pip install google-genai python-dotenv ipykernel
import os, json
from google import genai
from google.genai import types
try:
    from google.colab import userdata          # Colab: llave en Secrets
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:                            # VS Code: llave en el archivo .env
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)
 
# Elegir un modelo "flash-lite" disponible en su cuenta
excluir = ("tts", "image", "live", "audio", "embedding", "preview", "exp")
modelos = [m.name.replace("models/", "") for m in client.models.list()
           if "generateContent" in (m.supported_actions or [])
           and "gemini" in m.name
           and not any(x in m.name for x in excluir)]
MODELO = (sorted(m for m in modelos if "flash-lite" in m)
          or sorted(m for m in modelos if "flash" in m) or modelos)[-1]
print("Modelo:", MODELO)


Modelo: gemini-3.5-flash-lite


In [9]:
PREGUNTA = "Un cliente deja US$25 de margen al mes y el churn mensual es 4% ¿Cuánto vale (CLV)?"
r = client.models.generate_content(model=MODELO, contents=PREGUNTA)
print(r.text)


Para calcular el Valor de Vida del Cliente (**CLV** o *Customer Lifetime Value*), la fórmula más sencilla y utilizada cuando se tiene un margen y un porcentaje de cancelación (churn) constante es:

$$\text{CLV} = \frac{\text{Margen mensual}}{\text{Churn mensual}}$$

Aplicando los datos que proporcionas:

*   **Margen mensual:** US$ 25
*   **Churn mensual:** 4% (o 0.04 en formato decimal)

Cálculo:
$$\text{CLV} = \frac{25}{0.04} = 625$$

El valor del cliente (CLV) es **US$ 625**.


In [3]:
PREGUNTA = "Un cliente deja US$25 de margen al mes y el churn mensual es 4% y una tasa de descuento mensual del 0.8%. ¿Cuánto vale (CLV)?"
r = client.models.generate_content(model=MODELO, contents=PREGUNTA)
print(r.text)


Para calcular el Valor de Vida del Cliente (CLV - *Customer Lifetime Value*), utilizamos la fórmula estándar para flujos de caja periódicos con una tasa de churn y una tasa de descuento constantes:

$$\text{CLV} = \frac{\text{Margen mensual}}{\text{Churn mensual} + \text{Tasa de descuento mensual}}$$

**Datos proporcionados:**
*   Margen mensual ($M$) = US$ 25
*   Churn mensual ($r$) = $4\% = 0.04$
*   Tasa de descuento mensual ($d$) = $0.8\% = 0.008$

**Cálculo:**

$$\text{CLV} = \frac{25}{0.04 + 0.008}$$

$$\text{CLV} = \frac{25}{0.048}$$

$$\text{CLV} = 520.83$$

El valor del cliente (CLV) es **US$ 520.83**.


In [4]:
def calcular_clv(margen_mensual: float, churn_mensual: float) -> dict:
    """Calcula el CLV con la fórmula del curso:
    CLV = m × (1 − c) ÷ (c + d), con d = 0.008 (0.8% mensual).
    margen_mensual: margen por cliente al mes en US$.
    churn_mensual: proporción que se va cada mes (0.04 = 4%)."""
    m, c, d = margen_mensual, churn_mensual, 0.008
    return {"clv": round(m * (1 - c) / (c + d), 2)}
 
print(calcular_clv(25, 0.04))     # debe dar 500.0


{'clv': 500.0}


## ¿Por qué el LLM da tres respuestas distintas?

**La respuesta correcta es US$500.** Sin herramienta, el LLM elige su propia fórmula, y la fórmula cambia según cómo le preguntemos.

**Datos:** m = margen mensual = US$25 · c = churn mensual (clientes que se van cada mes) = 4% = 0.04 · d = tasa de descuento mensual = 0.8% = 0.008

| Lo que le dimos | Fórmula que usó | Cuenta | Resultado |
|---|---|---|---|
| Solo m y c | m ÷ c | 25 ÷ 0.04 | **US$625** |
| m, c y también d en la pregunta | m ÷ (c + d) | 25 ÷ 0.048 | **US$520.83** |
| La herramienta `calcular_clv` | m × (1 − c) ÷ (c + d) | 24 ÷ 0.048 | **US$500** |

**¿Qué le faltó al LLM?**
- **(1 − c):** en el curso aprendimos que solo se cuenta al 96% de clientes que se queda desde el primer mes. El LLM se lo saltó las dos veces.
- **+ d:** un dólar en el futuro vale menos que hoy. En la primera respuesta también se lo saltó.

**Conclusión:** dar más datos en la pregunta **no garantiza** la fórmula correcta. La herramienta sí, porque la fórmula está escrita en código. **El LLM explica, el código calcula.**

In [5]:
INSTRUCCIONES = """Eres un tutor de Marketing Analytics.
1. Para cualquier número, usa una herramienta. Nunca calcules de memoria.
2. Si falta un dato, pregunta; no lo supongas.
3. Explica en pasos cortos, muestra la fórmula y define cada sigla."""
CONFIG = types.GenerateContentConfig(system_instruction=INSTRUCCIONES,
    tools=[calcular_clv], temperature=0.2,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
r = client.models.generate_content(model=MODELO, contents=PREGUNTA,
                                   config=CONFIG)
for fc in r.function_calls or []:
    print("El LLM pide:", fc.name, dict(fc.args))


El LLM pide: calcular_clv {'churn_mensual': 0.04, 'margen_mensual': 25}


In [ ]:
HERRAMIENTAS = {"calcular_clv": calcular_clv}

def preguntar(pregunta, historial=None, max_pasos=5):
    historial = list(historial or []) + [types.Content(role="user", parts=[types.Part.from_text(text=pregunta)])]
    for paso in range(1, max_pasos + 1):
        r = client.models.generate_content(model=MODELO, contents=historial, config=CONFIG)
        historial.append(r.candidates[0].content)
        if not r.function_calls:                             # no pidió herramientas: respuesta final
            return r.text, historial
        partes = []
        for fc in r.function_calls:
            args = dict(fc.args or {})
            try:
                salida = HERRAMIENTAS[fc.name](**args)       # NUESTRO código ejecuta la herramienta
            except Exception as e:                           # baranda: si la pidió mal, no se cae
                salida = {"error": f"{type(e).__name__}: {e}. No inventes el dato: pídeselo al usuario."}
            print(f"🧠 Paso {paso}: pide {fc.name}({json.dumps(args)})  →  🔧 {salida}")
            partes.append(types.Part.from_function_response(name=fc.name, response={"resultado": salida}))
        historial.append(types.Content(role="user", parts=partes))
    return "Me detuve: llegué al máximo de pasos.", historial

respuesta, h = preguntar(PREGUNTA)
print("\n" + respuesta)


In [7]:
r1, h = preguntar("¿Cuánto vale un cliente si el churn mensual es 4%?")
print(r1)


Para calcular el valor del cliente (**CLV** o *Customer Lifetime Value*), necesito un dato adicional que no mencionaste:

* **¿Cuál es el margen mensual ($m$) que deja ese cliente en US$?**

---

Una vez que me des ese dato, utilizaremos la siguiente fórmula del curso:

$$\text{CLV} = \frac{m \times (1 - c)}{c + d}$$

**Definición de sigla:**
* **CLV**: *Customer Lifetime Value* (Valor de vida del cliente).
* **$m$**: Margen mensual que genera el cliente.
* **$c$**: Tasa de *churn* mensual (la proporción de clientes que se van cada mes, en este caso **$4\%$** o **$0.04$**).
* **$d$**: Tasa de descuento mensual fija del **$0.8\%$** (es decir, **$0.008$**).

Por favor, dime cuál es el margen mensual y calculamos el CLV paso a paso.


In [8]:
r2, h = preguntar("El margen es US$40.", historial=h)
print(r2)

¡Excelente! Ya tenemos todos los datos necesarios. Vamos a calcular el CLV paso a paso usando la fórmula del curso:

$$\text{CLV} = \frac{m \times (1 - c)}{c + d}$$

### Definición de siglas y valores:
* **CLV**: *Customer Lifetime Value* (Valor de vida del cliente).
* **$m$**: Margen mensual = **US$ 40**
* **$c$**: Churn mensual (proporción que se va cada mes) = **$4\%$** o **$0.04$**
* **$d$**: Tasa de descuento mensual fija = **$0.8\%$** o **$0.008$**

---

### Pasos del cálculo:

1. **Calculamos la retención y el numerador:**
   * Restamos $1 - c$:  
     $$1 - 0.04 = 0.96$$
   * Multiplicamos por el margen $m$:  
     $$40 \times 0.96 = 38.4$$

2. **Calculamos el denominador:**
   * Sumamos la tasa de churn ($c$) y la tasa de descuento ($d$):  
     $$0.04 + 0.008 = 0.048$$

3. **Dividimos el numerador entre el denominador (CLV):**
   * $$\text{CLV} = \frac{38.4}{0.048} = 800$$

El valor de este cliente (**CLV**) es de **US$ 800**.


# Demo D · El agente de retención: del riesgo a la oferta
Le damos al agente una **meta**, no una pregunta. Él decide los pasos:
1. **Predice el riesgo** con el modelo campeón de churn de DataRobot.
2. **Lee el CRM** (Customer Relationship Management, la base de clientes).
3. **Redacta ofertas**, una distinta para cada cliente.
4. Una **persona aprueba** las ofertas caras.
5. Se **envían** (aquí simulado: se guardan en un archivo, no sale ningún correo real).
6. Se **revisa** quién respondió y lo aprendido vuelve al siguiente ciclo.

Requisito: haber corrido los pasos 1 a 5 (client, MODELO, types, json, os).

### Paso 8 · Conectar con DataRobot (el modelo campeón de churn)

In [ ]:
# VS Code (terminal): pip install datarobot datarobot-predict pandas
# Colab (celda):      !pip -q install datarobot datarobot-predict
# Agreguen al archivo .env (en Colab, a Secrets):
#   DATAROBOT_API_TOKEN=su_llave_de_datarobot
#   DATAROBOT_ENDPOINT=https://app.datarobot.com/api/v2
#   DEPLOYMENT_ID=el_id_de_su_deployment_campeon
import json, os
import pandas as pd

USAR_DATAROBOT = True      # Plan B: pongan False si no tienen deployment; el riesgo sale de una regla simple

def secreto(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except ImportError:
        return os.getenv(nombre)

if USAR_DATAROBOT:
    import datarobot as dr
    from datarobot_predict.deployment import predict
    dr.Client(token=secreto("DATAROBOT_API_TOKEN"), endpoint=secreto("DATAROBOT_ENDPOINT"))
    DEPLOYMENT = dr.Deployment.get(secreto("DEPLOYMENT_ID"))
    print("Deployment:", DEPLOYMENT.label)
    print("Variables que pide el modelo:", [f["name"] for f in DEPLOYMENT.get_features()])

### Paso 9 · El CRM y una predicción de prueba

In [ ]:
URL_CRM = ("https://raw.githubusercontent.com/stanlybolanos/marketing-analytics-base-2026/"
           "main/Semana_3/Tarea/data/uplift_campaign.csv")
datos = pd.read_csv(URL_CRM)
RESPUESTAS = dict(zip(datos.customer_id, datos.responded_60d))   # se guardan aparte: el agente NO las ve
CRM = (datos[["customer_id", "recency_days", "frequency", "monetary", "email_open_rate", "tenure_days"]]
       .sample(200, random_state=7).reset_index(drop=True))
print(f"CRM: {len(CRM)} clientes")

if USAR_DATAROBOT:
    prueba = predict(DEPLOYMENT, CRM.head(3)).dataframe
    print("Columnas que devuelve el modelo:", prueba.columns.tolist())
    probas = [c for c in prueba.columns if c.endswith("_PREDICTION") and prueba[c].dtype.kind == "f"]
    COL_RIESGO = next((c for c in probas if "_1_" in c or "_True_" in c or "_Yes_" in c), probas[-1])
    print("Columna de probabilidad elegida:", COL_RIESGO)
# Revisen: ¿esa columna es la probabilidad de IRSE? Si es la de quedarse o responder, pongan True:
INVERTIR = False

### Paso 10 · Las herramientas del agente de retención

In [ ]:
OFERTAS = []               # bandeja de ofertas que propone el agente
LIMITE_DESCUENTO = 20      # baranda: % máximo que el código acepta
APROBACION_DESDE = 10      # desde este % decide una persona
_riesgo = None

def predecir_riesgo(n: int) -> dict:
    """Paso 1. Calcula la probabilidad de irse (riesgo de churn) de todos los clientes del CRM
    con el modelo campeón de DataRobot y devuelve los n de mayor riesgo.
    n: cuántos clientes devolver (máximo 20)."""
    global _riesgo
    n = max(1, min(int(n), 20))
    if _riesgo is None:
        if USAR_DATAROBOT:
            p = predict(DEPLOYMENT, CRM).dataframe[COL_RIESGO].to_numpy()
            _riesgo = 1 - p if INVERTIR else p
        else:   # Plan B: más días sin comprar y menos compras = más riesgo
            _riesgo = (CRM.recency_days.rank(pct=True) + (1 - CRM.frequency.rank(pct=True))).to_numpy() / 2
    tabla = CRM.assign(riesgo=_riesgo.round(3)).nlargest(n, "riesgo")
    return {"clientes": tabla[["customer_id", "riesgo"]].to_dict("records")}

def leer_crm(customer_ids: list[int]) -> dict:
    """Paso 2. Devuelve el historial de los clientes pedidos: días desde su última compra
    (recency_days), número de compras (frequency), gasto total en US$ (monetary),
    tasa de apertura de correos (email_open_rate) y antigüedad en días (tenure_days)."""
    filas = CRM[CRM.customer_id.isin([int(i) for i in customer_ids])]
    if filas.empty:
        return {"error": "Ninguno de esos clientes está en el CRM."}
    return {"historiales": filas.to_dict("records")}

def guardar_oferta(customer_id: int, descuento_pct: float, mensaje: str) -> dict:
    """Paso 3. Guarda la oferta de UN cliente.
    descuento_pct: descuento en % (por ejemplo 10).
    mensaje: el texto del correo, máximo 3 líneas, personalizado con su historial."""
    customer_id = int(customer_id)
    if customer_id not in set(CRM.customer_id):                       # baranda 1
        return {"error": f"El cliente {customer_id} no existe en el CRM. No inventes clientes."}
    if not 0 < descuento_pct <= LIMITE_DESCUENTO:                    # baranda 2
        return {"error": f"El descuento debe estar entre 1% y {LIMITE_DESCUENTO}%."}
    if any(o["customer_id"] == customer_id for o in OFERTAS):         # baranda 3
        return {"error": f"El cliente {customer_id} ya tiene oferta."}
    estado = "pendiente_aprobacion" if descuento_pct >= APROBACION_DESDE else "aprobada_automatica"
    OFERTAS.append({"customer_id": customer_id, "descuento_pct": descuento_pct,
                    "mensaje": mensaje, "estado": estado})
    return {"guardada": True, "estado": estado}

HERR_RETENCION = {f.__name__: f for f in (predecir_riesgo, leer_crm, guardar_oferta)}

### Paso 11 · El agente trabaja solo (pasos 1, 2 y 3)

In [ ]:
def ejecutar_agente(meta, config, herramientas, max_pasos=15):
    """El mismo ciclo del paso 5, pero con las herramientas y la configuración que le pasemos."""
    historial = [types.Content(role="user", parts=[types.Part.from_text(text=meta)])]
    for paso in range(1, max_pasos + 1):
        r = client.models.generate_content(model=MODELO, contents=historial, config=config)
        historial.append(r.candidates[0].content)
        if not r.function_calls:
            return r.text, historial
        partes = []
        for fc in r.function_calls:
            args = dict(fc.args or {})
            try:
                salida = herramientas[fc.name](**args)
            except Exception as e:                                      # baranda: no se cae
                salida = {"error": f"{type(e).__name__}: {e}"}
            print(f"🧠 Paso {paso}: {fc.name}({json.dumps(args, ensure_ascii=False)[:90]}) → {str(salida)[:90]}")
            partes.append(types.Part.from_function_response(name=fc.name, response={"resultado": salida}))
        historial.append(types.Content(role="user", parts=partes))
    return "Me detuve: llegué al máximo de pasos.", historial

INSTR_RETENCION = """Eres un agente de retención de clientes de una cafetería.
Trabajas solo, sin preguntarle nada al usuario.
1. Usa predecir_riesgo para encontrar a los clientes con más riesgo de irse.
2. Usa leer_crm UNA vez con todos esos clientes para ver su historial.
3. Para CADA cliente usa guardar_oferta: un descuento y un mensaje de máximo 3 líneas
   que mencione algo de su historial. Da más descuento solo a quien más ha gastado.
4. Nunca inventes clientes ni datos. Al terminar, resume cuántas ofertas guardaste."""

def configurar(instrucciones):
    return types.GenerateContentConfig(system_instruction=instrucciones,
        tools=list(HERR_RETENCION.values()), temperature=0.2,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

OFERTAS.clear()
resumen, _ = ejecutar_agente("Meta: retener a los 10 clientes con mayor riesgo de irse.",
                             configurar(INSTR_RETENCION), HERR_RETENCION)
print("\n" + resumen)
pd.DataFrame(OFERTAS)

### Paso 12 · La persona aprueba (paso 4) y se envía (paso 5, simulado)

In [ ]:
for o in OFERTAS:
    if o["estado"] == "pendiente_aprobacion":
        print(f"\nCliente {o['customer_id']} · {o['descuento_pct']}% de descuento\n{o['mensaje']}")
        o["estado"] = "aprobada" if input("¿Aprobar? (s/n): ").strip().lower() == "s" else "rechazada"

ENVIADAS = pd.DataFrame([o for o in OFERTAS if o["estado"] in ("aprobada", "aprobada_automatica")])
ENVIADAS.to_csv("bandeja_de_salida.csv", index=False)
print(f"\n{len(ENVIADAS)} ofertas 'enviadas' (guardadas en bandeja_de_salida.csv). "
      "No se envió ningún correo real.")

### Paso 13 · Revisar y ajustar (paso 6): lo aprendido vuelve al siguiente ciclo

In [ ]:
# Simulamos "7 días después" con las respuestas reales del experimento de Starbucks (responded_60d).
resultado = ENVIADAS.assign(respondio=ENVIADAS.customer_id.map(RESPUESTAS))
print(f"Respondió el {resultado.respondio.mean():.0%} de los clientes contactados")
print(resultado.groupby("descuento_pct").respondio.agg(["count", "mean"]))

r = client.models.generate_content(model=MODELO, contents=(
    "Estos son los resultados de una campaña de retención (respondio: 1 = sí, 0 = no):\n"
    + resultado[["customer_id", "descuento_pct", "respondio"]].to_string(index=False)
    + "\nEscribe 2 lecciones cortas para el siguiente ciclo. No inventes datos que no estén aquí."))
LECCIONES = r.text
print("\nLecciones para el siguiente ciclo:\n" + LECCIONES)

# El siguiente ciclo arranca con las lecciones dentro de las instrucciones (memoria a largo plazo):
INSTR_CICLO_2 = INSTR_RETENCION + "\nLecciones del ciclo anterior:\n" + LECCIONES
# Para correr el ciclo 2: OFERTAS.clear(); ejecutar_agente("Meta: …", configurar(INSTR_CICLO_2), HERR_RETENCION)